# PyTorch Tutorial - Neural Networks
Prof. Dorien Herremans, with many thanks to Nelson Lui for the base text.

**To edit the notebook**:

There are two ways to edit the notebook.

You can either open it in the "playground", where you can change and run cells. After closing the tab, your changes will be lost. To do so, press "File" > "Open in playground".

Alternatively, you can make a copy of this notebook to your own Google Drive account through "File" > "Save a copy in Drive..."

**Activating the GPU on Colab**:

Colab now gives you 12 hours of free GPU time (before you have to request a new node).
Simply select "GPU" in the Accelerator drop-down in Notebook Settings (either through the Edit menu or the command palette at cmd/ctrl-shift-P).

# Setting up the notebook on colab

Let's check if we are using the GPU environment and cuda is installed:

In [1]:
# Import PyTorch and other libraries
import torch
import numpy as np
from tqdm import tqdm

print("PyTorch version:")
print(torch.__version__)
print("GPU Detected:")
print(torch.cuda.is_available())

#defining a shortcut function for later:
import os
using_GPU = os.path.exists('/opt/bin/nvidia-smi')

PyTorch version:
2.10.0+cu128
GPU Detected:
True


# Computation Graphs

A computation graph is simply a way to define a sequence of operations to go from input to model output.

You can think of the nodes in the graph as representing operations, and the edges in the graph represent tensors going in and out.

For example, say we wanted to build a linear regression model. This has the form $\hat y = Wx + b$.

In this equation, $x$ is our input, $W$ is a learned weight matrix, $b$ is a learned bias, and $\hat y$ is the predicted output.

As a computation graph, this looks like:

![Linear Regression Computation Graph](https://imgur.com/IcBhTjS.png)

When implementing deep learning models, you're basically designing and specifying computation graphs. It's a bit like playing with Legos in that you're stringing together a bunch of blocks (the operations) to achieve a final desired output.

# The building blocks of deep learning models

`torch.nn` makes it easy to build neural nets by providing functions for specifying arbitrary computation graphs and abstractions for putting them all together. We'll start by covering a few classes in the `torch.nn` module that form basic building blocks of many deep learning applications.

The classes below are all callable, so you can use them with `outputs = YourDeepLearningBlock(its_inputs)`

In [2]:
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable

## Linear Layers (Affine Transforms)

A linear layer (also known as an affine transform) defines a function:

$$f(x) = Wx + b$$

This linear transform is a core part of deep learning. $W$ and $b$ are the parameters of this layer, where $W$ is a learned weight matrix and $b$ is a learned bias vector.

`nn.Linear()` takes two construction parameters: the dimensionality of the input and the dimensionality of the desired output.

In [3]:
# Create a Linear layer. Input should have 5 dimensions, output will have 3.
lin = nn.Linear(5, 3)
# Data is a matrix of shape (2, 5). Can we use the linear layer on it?
data = torch.randn(2, 5)

# Yes! Running the data matrix through the layer outputs shape (2, 3).
print(lin(data))

tensor([[-1.3857, -0.1046,  0.3579],
        [-1.2963, -0.2067,  0.5604]], grad_fn=<AddmmBackward0>)


In [4]:
# What about a matrix of shape (2, 4, 5)?
data = torch.randn(2, 4, 5)
# This works as well! As long as the last dimension is the specified
# input dimension to the Linear layer, you're good.
# Output shape: (2, 4, 3)
print(lin(data))

tensor([[[ 0.0898, -0.1679, -0.4608],
         [ 0.3392,  0.3353, -1.4706],
         [ 0.3520, -1.0913, -1.1167],
         [-0.8881, -0.4661, -0.3550]],

        [[ 0.4576,  0.1621, -1.0643],
         [-0.4520, -0.5925, -0.9433],
         [-0.8692,  0.1134,  0.1740],
         [-0.3339, -0.3661,  0.3257]]], grad_fn=<ViewBackward0>)


In [5]:
# But (5, 2) is an incompatible shape (uncomment and run to see error)
data = torch.randn(5, 2)
# print(lin(data))

In [6]:
# But we can transpose it using t()!
# Now its shape (2, 5) and all is fine.
print(lin(data.t()))

tensor([[-0.8170,  0.7149,  0.8734],
        [-1.7243,  0.5310,  1.8440]], grad_fn=<AddmmBackward0>)


## Nonlinearities / Activation Functions

Since composing linear transformations gives you a linear transformation, we don't gain any representational power by just chaining `Linear` layers.

In deep learning, we add nonlinearities after our Linear transforms, which lets us build more powerful models.

PyTorch comes with a veritable zoo of nonlinearities.

In [7]:
data = torch.randn(2, 3)
print(data)

# Nonlinearities are layers too!
relu = nn.ReLU()
print(relu)
print(relu(data))

tanh = nn.Tanh()
print(tanh)
print(tanh(data))

sigmoid = nn.Sigmoid()
print(sigmoid)
print(sigmoid(data))

tensor([[ 0.3250, -0.6588, -1.6707],
        [ 1.1521, -0.3182,  2.0807]])
ReLU()
tensor([[0.3250, 0.0000, 0.0000],
        [1.1521, 0.0000, 2.0807]])
Tanh()
tensor([[ 0.3140, -0.5775, -0.9316],
        [ 0.8185, -0.3079,  0.9693]])
Sigmoid()
tensor([[0.5805, 0.3410, 0.1583],
        [0.7599, 0.4211, 0.8890]])


If you'd prefer to not create a class for the nonlinearity, you can also call it functionally as below:

In [8]:
data = torch.randn(2, 3)
print(data)

# Nonlinearities can also be used functionally, with no need to create a class!
print("ReLu:")
print(torch.relu(data))

print("tanh:")
print(torch.tanh(data))

print("Sigmoid:")
print(torch.sigmoid(data))

tensor([[-1.7931, -0.9059, -0.2080],
        [-0.6325,  0.8674,  0.0748]])
ReLu:
tensor([[0.0000, 0.0000, 0.0000],
        [0.0000, 0.8674, 0.0748]])
tanh:
tensor([[-0.9461, -0.7191, -0.2051],
        [-0.5598,  0.7000,  0.0746]])
Sigmoid:
tensor([[0.1427, 0.2878, 0.4482],
        [0.3469, 0.7042, 0.5187]])


## Dropout

Dropout is used to regularize our models by randomly setting some outputs to 0.

This helps to prevent overfitting by encouraging the model to look beyond specific spurious patterns and find features that generalize.

**Note that we should only apply dropout during training!**

In [9]:
data = torch.randn(2, 3)
print(data)

# Create a Dropout layer and call it on input
# Here, the probability of zeroing an element is 0.5
dropout = nn.Dropout(0.5)
print(dropout)
print(dropout(data))

# Use dropout functionally, training=False by default so no change.
print("Functional dropout, training=False")
print(F.dropout(data, 0.5, training=False))

# Set training=True, so things are dropped out
print("Functional dropout, training=True")
print(F.dropout(data, 0.5, training=True))

tensor([[ 0.5061,  0.6329,  1.2201],
        [-0.2638,  0.3156,  0.9911]])
Dropout(p=0.5, inplace=False)
tensor([[ 0.0000,  0.0000,  2.4403],
        [-0.5275,  0.6312,  0.0000]])
Functional dropout, training=False
tensor([[ 0.5061,  0.6329,  1.2201],
        [-0.2638,  0.3156,  0.9911]])
Functional dropout, training=True
tensor([[ 0.0000,  0.0000,  0.0000],
        [-0.5275,  0.6312,  0.0000]])


# Structuring PyTorch models

At the highest level, `nn.Module` defines what most would refer to as a "model". It's a convenient way for encapsulating the trainable parameters of a model or a component of your model, and subclassing this class gives you Python functions for moving your model to the GPU, saving it, loading it etc.

When you're building your own model, you're going to subclass `nn.Module`. Critically, you also need to override the `__init__()` and `forward()` functions.

*   In `__init__()`, you should take arguments that modify how the model runs (e.g. # of layers, # of hidden units, output sizes). You'll also set up most of the layers that you use in the forward pass here.
*   In `forward()`, you define the "forward pass" of your model, or the operations needed to transform input to output. **You can use any of the Tensor operations in the forward pass.**



### Logistic Regression

is a simple example of how to make a Module. It is also an example of a neuron. Let's build a logistic regression model.

Logistic regression takes an input $x$ and applies a linear transform to squash the input down to a probability distribution over the number of classification classes. If you recall from the lecture, we start with a linear regression model based on the input variables, which is then put into a logistic sigmoid function. As a module, this looks like:

In [10]:
import torch.nn as nn
import torch.nn.functional as F

class LogisticRegression(nn.Module):
  # input_size: Dimensionality of input feature vector.
  # num_classes: The number of classes in the classification problem.
  def __init__(self, input_size, num_classes):
    # Always call the superclass (nn.Module) constructor first!
    super(LogisticRegression, self).__init__()
    # Set up the linear transform
    self.linear = nn.Linear(input_size, num_classes)
    # I do not yet include the sigmoid activation after the linear
    # layer because our loss function will include this as you will see later

  # Forward's sole argument is the input.
  # input is of shape (batch_size, input_size)
  def forward(self, x):
    # Apply the linear transform.
    # out is of shape (batch_size, num_classes).
    out = self.linear(x)
    out = torch.sigmoid(out)
    # Softmax the out tensor to get a log-probability distribution
    # over classes for each example.
    return out

### Feed-forward neural net

To improve upon the logistic regression model, we can add some intermediate layers (called hidden layers), nonlinearities, and dropout for regularization. This is essentially a multi-layer feed forward neural net, and it's implementation as a module is outlined below:

In [11]:
class FeedForwardNN(nn.Module):
  # input_size: Dimensionality of input feature vector.
  # num_classes: The number of classes in the classification problem.
  # num_hidden: The number of hidden (intermediate) layers to use.
  # hidden_dim: The size of each of the hidden layers.
  # dropout: The proportion of units to drop out after each layer.
  def __init__(self, input_size, num_classes, num_hidden, hidden_dim, dropout):
    # Always call the superclass (nn.Module) constructor first!
    super(FeedForwardNN, self).__init__()

    # Set up the hidden layers.
    assert num_hidden > 0
    # A special ModuleList to store our hidden layers.
    self.hidden_layers = nn.ModuleList([])
    # First hidden layer maps from input_size -> num_hidden.
    self.hidden_layers.append(nn.Linear(input_size, hidden_dim))
    # Subsequent hidden layers map from num_hidden -> num_hidden.
    # Note that they can map to any dimensionality --- as long as the final
    # output is a distribution over your classes!
    for i in range(num_hidden - 1):
      self.hidden_layers.append(nn.Linear(hidden_dim, hidden_dim))

    # Set up the dropout layer.
    self.dropout = nn.Dropout(dropout)

    # Set up the final transform to a distribution over classes.
    self.output_projection = nn.Linear(hidden_dim, num_classes)

    # Set up the nonlinearity to use between layers.
    self.nonlinearity = nn.ReLU()

  # Forward's sole argument is the input.
  # input is of shape (batch_size, input_size)
  def forward(self, x):
    # Apply the hidden layers, nonlinearity, and dropout.
    for hidden_layer in self.hidden_layers:
      x = hidden_layer(x)
      x = self.dropout(x)
      x = self.nonlinearity(x)

    # Output layer: project x to a distribution over classes.
    out = self.output_projection(x)

    # Softmax the out tensor to get a log-probability distribution
    # over classes for each example.
    out_distribution = F.log_softmax(out, dim=-1)
    return out_distribution

# Training PyTorch models: Losses and Optimizers

By now, we've learned how to construct models in PyTorch. In this section, we'll go over how to calculate your model's loss and how to optimize the parameters to minimize the loss.

## Loss Functions

Intuitively, loss functions serve to tell your model how poorly it's doing --- the purpose of training is to adjust the weights of our model to minimize the loss.

A loss function takes a true output $y$ and a model-predicted output $\hat y$ and calculates the loss. If $y = \hat y$, our model produced the correct output and thus our loss is 0. The further our predicted $\hat y$ from the true $y$, the higher our loss is.

PyTorch comes with a large collection of loss functions. The most commonly used loss for classification is negative log likelihood (`nn.NLLLoss` or the very related `nn.CrossEntropyLoss`). The difference between `nn.NLLLoss` and `nn.CrossEntropyLoss` for classification problems is that `nn.NLLLoss` expects the output to be log-softmax normalized, which is easy to do with the `nn.LogSoftmax` layer. On the other hand `nn.CrossEntropyLoss`, automatically applies the log-softmax --- you can think of it as `nn.LogSoftmax` + `nn.NLLLoss`. Which to use depends on whether you want to add the extra `nn.LogSoftmax` to your model's `forward()`.

A common loss used for regression problems is the mean squared error (`nn.MSELoss`).

Here's a usage example of the `CrossEntropyLoss`.

In [12]:
# 3 examples, unnormalized scores over 4 classes.
model_output = torch.rand(3, 4, requires_grad = True)

# The correct labels.
targets = torch.LongTensor([1, 0, 3])

# CrossEntropyLoss
cross_entropy = nn.CrossEntropyLoss()
# Loss, averaged across all 3 batch elements.
# Can call this functionally: avg_loss = F.cross_entropy(model_output, targets)
avg_loss = cross_entropy(model_output, targets)
print("CrossEntropyLoss averaged across all 3 batch elements:")
print(avg_loss)

# Backpropagate wrt avg_loss
avg_loss.backward()
# Print out the gradients of model_output
print("Gradients of model_output")
print(model_output.grad)

CrossEntropyLoss averaged across all 3 batch elements:
tensor(1.5997, grad_fn=<NllLossBackward0>)
Gradients of model_output
tensor([[ 0.1374, -0.2751,  0.0803,  0.0574],
        [-0.2503,  0.0852,  0.0871,  0.0779],
        [ 0.0579,  0.0909,  0.1215, -0.2703]])


And here's a snippet showing that `LogSoftmax` + `NLLLoss` is the same as `CrossEntropyLoss`.

In [13]:
nll = nn.NLLLoss()
log_softmax_model_output = F.log_softmax(model_output, dim=-1)
# Loss, averaged across all 3 batch elements.
# Can call this functionally: avg_loss = F.nll_loss(model_output, targets)
avg_loss = nll(log_softmax_model_output, targets)
print("Negative-Log Likelihood averaged across all 3 batch elements:")
print(avg_loss)

Negative-Log Likelihood averaged across all 3 batch elements:
tensor(1.5997, grad_fn=<NllLossBackward0>)


## Optimizers

Now that we can calculate the loss and backpropagate through our model (with `.backward()`), we can update the weights and try to reduce the loss!

PyTorch includes a variety of optimizers that do exactly this, from the standard SGD to more recent techniques like Adam and RMSProp.

At construction, PyTorch parameters take the parameters to optimize. When we run an input through our model, calculate the loss, and backpropagate, the gradients are automatically stored in the parameters (since they're all `Variables`). With these gradients, the optimizer can update the weights.

Optimizers live in the `torch.optim` module.

In [14]:
import torch.optim as optim

To get the parameters of our model, we can just call `.parameters()` on a `Module`. Below, we create an instance of our previously-defined feed forward neural network and get its parameters.

In [15]:
input_size = 784
num_classes = 10
num_hidden = 2
hidden_dim = 50
dropout = 0.2
ffnn_clf = FeedForwardNN(input_size, num_classes, num_hidden,
                         hidden_dim, dropout)
print(ffnn_clf)

parameters = ffnn_clf.parameters()

print("Shapes of model parameters:")
print([x.size() for x in list(parameters)])

FeedForwardNN(
  (hidden_layers): ModuleList(
    (0): Linear(in_features=784, out_features=50, bias=True)
    (1): Linear(in_features=50, out_features=50, bias=True)
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (output_projection): Linear(in_features=50, out_features=10, bias=True)
  (nonlinearity): ReLU()
)
Shapes of model parameters:
[torch.Size([50, 784]), torch.Size([50]), torch.Size([50, 50]), torch.Size([50]), torch.Size([10, 50]), torch.Size([10])]


Now to create an optimizer for this model, we construct a optimizer class and pass it the parameters of the model: stochastic gradient descend.

In [16]:
ffnn_optim = optim.SGD(ffnn_clf.parameters(), lr=0.5)

Let's try using our optimizer to take a gradient update on our model! We'll generate a few random examples, and run them through our model (the forward pass).

In [17]:
# Make some fake data for our model.
# 5 examples in the batch, each example has 784 features.
sample_input = torch.randn(5, 784)
# Multilabel classification, 10 possible classes.
sample_labels = torch.LongTensor([0, 3, 9, 6, 2])

# Run the sample_input through ffnn_clf to get a distribution
# over our classes
sample_predictions = ffnn_clf(sample_input)
print("Predicted distribution over classes: ")
print(sample_predictions)
print("Target Labels:")
print(sample_labels)

Predicted distribution over classes: 
tensor([[-2.3642, -2.2756, -2.3484, -2.2230, -2.1484, -2.3850, -2.3420, -2.3451,
         -2.3888, -2.2352],
        [-2.3408, -2.3918, -2.2122, -2.1758, -2.2123, -2.4137, -2.2391, -2.4897,
         -2.4406, -2.1733],
        [-2.3428, -2.5271, -2.4172, -2.0947, -2.3169, -2.2551, -2.1907, -2.6893,
         -2.3066, -2.0483],
        [-2.3393, -2.4907, -2.3718, -2.2354, -2.3994, -2.3677, -2.1490, -2.3792,
         -2.1978, -2.1563],
        [-2.3953, -2.2850, -2.2233, -2.2115, -2.3054, -2.3232, -2.4179, -2.3835,
         -2.3646, -2.1515]], grad_fn=<LogSoftmaxBackward0>)
Target Labels:
tensor([0, 3, 9, 6, 2])


Now let's calculate the loss of our model on these examples.

In [18]:
nll_loss = F.nll_loss(sample_predictions, sample_labels)
print("Average NLL Loss:")
print(nll_loss)

Average NLL Loss:
tensor(2.1921, grad_fn=<NllLossBackward0>)


Let's print the gradients of one of the parameter matrices in our model, to ensure it's `None`. We haven't done backprop yet, so there shouldn't be any gradients.

In [19]:
print(list(ffnn_clf.parameters())[0].grad)

None


Now we can backpropagate with respect to the loss to calculate the gradients for the parameters of our model with `.backward()`. It's also good practice to call `optimizer.zero_grad()` before `loss.backwards()`, which ensures that the gradients are reset to 0 before backprop.

In [20]:
ffnn_optim.zero_grad()
nll_loss.backward()

Let's check our gradients now...

In [21]:
print(list(ffnn_clf.parameters())[0].grad)

tensor([[-1.8019e-04,  2.9186e-03, -5.5557e-03,  ...,  3.5555e-04,
          1.2799e-03, -1.5491e-03],
        [-4.3337e-04,  7.0195e-03, -1.3362e-02,  ...,  8.5514e-04,
          3.0783e-03, -3.7258e-03],
        [ 2.8974e-02,  2.2701e-02, -7.1217e-03,  ..., -2.0101e-03,
         -1.2097e-03, -3.9025e-02],
        ...,
        [ 4.0954e-03, -6.1422e-03,  4.3538e-03,  ...,  1.6093e-02,
         -1.3401e-02, -8.5846e-03],
        [-5.1475e-03, -7.4302e-04,  1.2395e-02,  ..., -2.4489e-02,
          1.5147e-02,  1.7212e-02],
        [ 8.8461e-03,  2.9123e-03,  5.5760e-03,  ...,  3.0811e-03,
         -8.5283e-04, -7.2609e-05]])


Now that we have gradients for each of our parameters, we can update them by using `optimizer.step()`.

In [22]:
# save the old value of the parameter for comparison later
old_parameter = list(ffnn_clf.parameters())[0].data.clone()

# Make a gradient update with our optimizer
ffnn_optim.step()

new_parameter = list(ffnn_clf.parameters())[0].data

print("Difference between weight matrix before and after update:")
print(old_parameter - new_parameter)


Difference between weight matrix before and after update:
tensor([[-9.0094e-05,  1.4593e-03, -2.7779e-03,  ...,  1.7778e-04,
          6.3996e-04, -7.7457e-04],
        [-2.1669e-04,  3.5097e-03, -6.6811e-03,  ...,  4.2757e-04,
          1.5392e-03, -1.8629e-03],
        [ 1.4487e-02,  1.1350e-02, -3.5608e-03,  ..., -1.0050e-03,
         -6.0483e-04, -1.9512e-02],
        ...,
        [ 2.0477e-03, -3.0711e-03,  2.1769e-03,  ...,  8.0463e-03,
         -6.7004e-03, -4.2923e-03],
        [-2.5737e-03, -3.7151e-04,  6.1974e-03,  ..., -1.2244e-02,
          7.5733e-03,  8.6060e-03],
        [ 4.4231e-03,  1.4561e-03,  2.7880e-03,  ...,  1.5406e-03,
         -4.2642e-04, -3.6305e-05]])


If you're familiar with the SGD update rule, you know that:

$$ \theta^{t+1} = \theta^{t} - \left( \eta \cdot \nabla L \left(\theta^{t} \right) \right)$$

Where $\theta^{t}$ is the weight at time $t$, $\eta$ is the learning rate, $\nabla L(\theta^{t})$ is the gradient. Since $\eta = 0.5$, it makes perfect sense that the difference between the weight vectors printed above is exactly half of the gradient.

# Example: Classification on FashionMNIST

Let's use the `FeedForwardNN` model we built earlier to do a simple classification task! This example is meant to be an annotated walkthrough of how to build, train, and evaluate a model in PyTorch. We'll use the [FashionMNIST dataset](https://github.com/zalandoresearch/fashion-mnist), where we are tasked with classifying black and white images of clothes into 10 different classes.

## Loading Data

We'll start by loading the data with `torchvision` --- knowing how to use torchvision isn't the point of this tutorial, so it's relatively unannotated.

In [23]:
!pip install torchvision==0.10.0 #note: you can find compatible torch/torchvision versions here: https://github.com/pytorch/vision#installation
import torchvision
from torchvision.datasets import FashionMNIST

train_dataset = FashionMNIST(root='./torchvision-data',
                             train=True,
                             transform=torchvision.transforms.ToTensor(),
                             download=True)

test_dataset = FashionMNIST(root='./torchvision-data', train=False,
                            transform=torchvision.transforms.ToTensor())

ERROR: Ignored the following yanked versions: 0.1.6, 0.1.7, 0.1.8, 0.1.9, 0.2.0, 0.2.1, 0.2.2, 0.2.2.post2, 0.2.2.post3
ERROR: Could not find a version that satisfies the requirement torchvision==0.10.0 (from versions: 0.17.0, 0.17.1, 0.17.2, 0.18.0, 0.18.1, 0.19.0, 0.19.1, 0.20.0, 0.20.1, 0.21.0, 0.22.0, 0.22.1, 0.23.0, 0.24.0, 0.24.1, 0.25.0)
ERROR: No matching distribution found for torchvision==0.10.0


`train_dataset` and `test_dataset` are both subclasses of PyTorch's `torch.utils.data.Dataset`. The main benefit of subclassing this abstract class is that we can use `torch.utils.data.DataLoader`s to handle batching our examples and iterating over them. We'll create `DataLoader`s for our datasets now.

In [24]:
from torch.utils.data import DataLoader

# Data-related hyperparameters
batch_size = 64

# Set up a DataLoader for the training dataset.
train_dataloader = DataLoader(
    dataset=train_dataset, batch_size=batch_size, shuffle=True)

# Set up a DataLoader for the test dataset.
test_dataloader = DataLoader(
    dataset=test_dataset, batch_size=batch_size)

Let's take a look at what's inside our datasets. `torch.utils.data.Dataset`s are indexable, so we can easily peek inside.

In [25]:
# Print the first training example
print(train_dataset[0])

(tensor([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000, 0.0510,
          0.2863, 0.0000, 0.0000, 0.0039, 0.0157, 0.0000

From this output, we can see the dataset elements are tuple of `(data_tensor, label)`. `data_tensor` is a `FloatTensor` of shape `(1, 28, 28)` (since the image is 28x28), and `label` is an integer from 0 to 9 (since there are 10 classes in the data).

Let's similarly look at what the `DataLoader` produces.

In [26]:
list(train_dataloader)[0]

[tensor([[[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           ...,
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]]],
 
 
         [[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0039, 0.0000],
           ...,
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0078],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]]],
 
 
         [[[0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
           [0.0000

As we can see, the `DataLoader` groups examples into batches of size `batch_size` (64 by default in the code above). Thus, the shape of the returned tensor is `(64, 1, 28, 28)`, since we essentially stacked `batch_size` examples together. Similarly, `labels` is now a `LongTensor` of size `batch_size`.

Note that the label for a single example was a Python `int` --- the dataloader automatically grouped them into a `LongTensor` of the appropriate size.

## Building our model

Now we can construct a `FeedForwardNN` instance that we'll train. Each FashionMNIST example is `28x28`, so we get it as a Tensor of shape `(28, 28)`.

We'll flatten out each example to a vector of size `(784,)` for compatibility with our model.

In [27]:
# Hyperparameters of our model.
num_hidden = 2
hidden_dim = 512
dropout = 0.2

fashionmnist_ffnn_clf = FeedForwardNN(input_size=784, num_classes=10,
                                      num_hidden=num_hidden,
                                      hidden_dim=hidden_dim, dropout=dropout)
print(fashionmnist_ffnn_clf)

FeedForwardNN(
  (hidden_layers): ModuleList(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): Linear(in_features=512, out_features=512, bias=True)
  )
  (dropout): Dropout(p=0.2, inplace=False)
  (output_projection): Linear(in_features=512, out_features=10, bias=True)
  (nonlinearity): ReLU()
)


## **Test Case 1: Construct Your Neural Network**

For a **classification problem with 6 classes**, construct a neural network with **three hidden layers** that outputs the **probability distribution over all classes**.

The **input dimension** is **1024**. The network architecture should be defined as follows:

- First hidden layer: 1024 units  
- Second hidden layer: 512 units  
- Third hidden layer: 128 units  

Name the neural network **`MineFeedForwardNN`**. After defining the model, **instantiate it and print the network architecture**.

In [38]:
import torch.nn as nn
import torch.nn.functional as F

class MineFeedForwardNN(nn.Module):
  # input_size: Dimensionality of input feature vector.
  # num_classes: The number of classes in the classification problem.
  # dropout_rate: The proportion of units to drop out after each hidden layer.
  def __init__(self, input_size, num_classes, dropout_rate=0.2):
    # Always call the superclass (nn.Module) constructor first!
    super(MineFeedForwardNN, self).__init__()

    # Define the hidden layers with specified dimensions
    self.hidden_layer1 = nn.Linear(input_size, 1024)
    self.hidden_layer2 = nn.Linear(1024, 512)
    self.hidden_layer3 = nn.Linear(512, 128)

    # Set up the dropout layer.
    self.dropout = nn.Dropout(dropout_rate)

    # Set up the final transform to a distribution over classes.
    self.output_projection = nn.Linear(128, num_classes)

    # Set up the nonlinearity to use between layers.
    self.nonlinearity = nn.ReLU()

  # Forward's sole argument is the input.
  # input is of shape (batch_size, input_size)
  def forward(self, x):
    # Apply first hidden layer, dropout, and nonlinearity
    x = self.hidden_layer1(x)
    x = self.dropout(x)
    x = self.nonlinearity(x)

    # Apply second hidden layer, dropout, and nonlinearity
    x = self.hidden_layer2(x)
    x = self.dropout(x)
    x = self.nonlinearity(x)

    # Apply third hidden layer, dropout, and nonlinearity
    x = self.hidden_layer3(x)
    x = self.dropout(x)
    x = self.nonlinearity(x)

    # Output layer: project x to a distribution over classes.
    out = self.output_projection(x)

    # For CrossEntropyLoss, the model should output raw logits (unnormalized scores).
    # CrossEntropyLoss internally applies LogSoftmax and NLLLoss.
    return out

# Instantiate the network as specified in the problem statement
input_dim = 1024
num_output_classes = 6
mine_ffnn_clf = MineFeedForwardNN(input_dim, num_output_classes)

# Print the network architecture
print(mine_ffnn_clf)

MineFeedForwardNN(
  (hidden_layer1): Linear(in_features=1024, out_features=1024, bias=True)
  (hidden_layer2): Linear(in_features=1024, out_features=512, bias=True)
  (hidden_layer3): Linear(in_features=512, out_features=128, bias=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (output_projection): Linear(in_features=128, out_features=6, bias=True)
  (nonlinearity): ReLU()
)


In [29]:
# Get the parameters of your MineFeedForwardNN model
mine_ffnn_parameters = mine_ffnn_clf.parameters()

print("Shapes of MineFeedForwardNN model parameters:")
print([x.size() for x in list(mine_ffnn_parameters)])

Shapes of MineFeedForwardNN model parameters:
[torch.Size([1024, 1024]), torch.Size([1024]), torch.Size([512, 1024]), torch.Size([512]), torch.Size([128, 512]), torch.Size([128]), torch.Size([6, 128]), torch.Size([6])]


If we're using a GPU, we'll move the model to the GPU which should speed up training. We do this with the same `.cuda()` method we used for Tensors.

In [30]:
if using_GPU:
  fashionmnist_ffnn_clf = fashionmnist_ffnn_clf.cuda()

# Check if the Module is on GPU by checking if a parameter is on GPU
print("Model on GPU?:")
print(next(fashionmnist_ffnn_clf.parameters()).is_cuda)

Model on GPU?:
True


## Construct other classes we need for training: loss and optimizer

Now, we'll set up a criterion for calculating the loss and an Optimizer for updating our parameters.

In [31]:
# Set up criterion for calculating loss
nll_criterion = nn.NLLLoss()

lr = 0.1
momentum = 0.9
# Set up an optimizer for updating the parameters of fashionmnist_ffnn_clf
ffnn_optimizer = optim.SGD(fashionmnist_ffnn_clf.parameters(),
                           lr=lr, momentum=momentum)

## Train the model!

Now, we'll implement the procedure to train the model --- this is typically called the "train loop" since we loop over our batches, performing the forward pass, calculating a loss, backpropping, and then updating our parameters. This is the bulk of the code necessary to train the model.

This block looks pretty long, but that's mostly because of the comments :)

In [32]:
# Number of epochs (passes through the dataset) to train the model for.
num_epochs = 10

# A counter for the number of gradient updates we've performed.
num_iter = 0

# Iterate `num_epochs` times.
for epoch in range(num_epochs):
  print("Starting epoch {}".format(epoch + 1))
  # Iterate over the train_dataloader, unpacking the images and labels
  for (images, labels) in train_dataloader:
    # Reshape images from (batch_size, 1, 28, 28) to (batch_size, 784), since
    # that's what our model expects. Remember that -1 does shape inference!
    reshaped_images = images.view(-1, 784)

    # Wrap reshaped_images and labels in Variables,
    # since we want to calculate gradients and backprop.
    reshaped_images = Variable(reshaped_images)
    labels = Variable(labels)

    # If we're using the GPU, move reshaped_images and labels to the GPU.
    if using_GPU:
      reshaped_images = reshaped_images.cuda()
      labels = labels.cuda()

    # Run the forward pass through the model to get predicted log distribution.
    # predicted shape: (batch_size, 10) (since there are 10 classes)
    predicted = fashionmnist_ffnn_clf(reshaped_images)

    # Calculate the loss
    batch_loss = nll_criterion(predicted, labels)

    # Clear the gradients as we prepare to backprop.
    ffnn_optimizer.zero_grad()

    # Backprop (backward pass), which calculates gradients.
    batch_loss.backward()

    # Take a gradient step to update parameters.
    ffnn_optimizer.step()

    # Increment gradient update counter.
    num_iter += 1

    # Calculate test set loss and accuracy every 500 gradient updates
    # It's standard to have this as a separate evaluate function, but
    # we'll place it inline for didactic purposes.
    if num_iter % 500 == 0:
      # Set model to eval mode, which turns off dropout.
      fashionmnist_ffnn_clf.eval()
      # Counters for the num of examples we get right / total num of examples.
      num_correct = 0
      total_examples = 0
      total_test_loss = 0

      # Iterate over the test dataloader
      for (test_images, test_labels) in test_dataloader:
        # Reshape images from (batch_size, 1, 28, 28) to (batch_size, 784) again
        reshaped_test_images = test_images.view(-1, 784)

        # Wrap test data in Variable, like we did earlier.
        # We set volatile=True bc we don't need history; speeds up inference.
        reshaped_test_images = Variable(reshaped_test_images, volatile=True)
        test_labels = Variable(test_labels, volatile=True)

        # If we're using the GPU, move tensors to the GPU.
        if using_GPU:
          reshaped_test_images = reshaped_test_images.cuda()
          test_labels = test_labels.cuda()

        # Run the forward pass to get predicted distribution.
        predicted = fashionmnist_ffnn_clf(reshaped_test_images)

        # Calculate loss for this test batch. This is averaged, so multiply
        # by the number of examples in batch to get a total.
        total_test_loss += nll_criterion(
            predicted, test_labels).data * test_labels.size(0)

        # Get predicted labels (argmax)
        # We need predicted.data since predicted is a Variable, and torch.max
        # expects a Tensor as input. .data extracts Tensor underlying Variable.
        _, predicted_labels = torch.max(predicted.data, 1)

        # Count the number of examples in this batch
        total_examples += test_labels.size(0)

        # Count the total number of correctly predicted labels.
        # predicted == labels generates a ByteTensor in indices where
        # predicted and labels match, so we can sum to get the num correct.
        num_correct += torch.sum(predicted_labels == test_labels.data)
      accuracy = 100 * num_correct / total_examples
      average_test_loss = total_test_loss / total_examples
      print("Iteration {}. Test Loss {}. Test Accuracy {}.".format(
          num_iter, average_test_loss, accuracy))
      # Set the model back to train mode, which activates dropout again.
      fashionmnist_ffnn_clf.train()

Starting epoch 1


/tmp/ipykernel_16291/1347851882.py:63: UserWarning: volatile was removed and now has no effect. Use `with torch.no_grad():` instead.
  reshaped_test_images = Variable(reshaped_test_images, volatile=True)
/tmp/ipykernel_16291/1347851882.py:64: UserWarning: volatile was removed and now has no effect. Use `with torch.no_grad():` instead.
  test_labels = Variable(test_labels, volatile=True)


Iteration 500. Test Loss 0.6546705961227417. Test Accuracy 75.9000015258789.
Starting epoch 2
Iteration 1000. Test Loss 0.6056337952613831. Test Accuracy 75.41999816894531.
Iteration 1500. Test Loss 0.5878776907920837. Test Accuracy 79.04999542236328.
Starting epoch 3
Iteration 2000. Test Loss 0.509900689125061. Test Accuracy 80.47999572753906.
Iteration 2500. Test Loss 0.5079975128173828. Test Accuracy 81.38999938964844.
Starting epoch 4
Iteration 3000. Test Loss 0.4806784689426422. Test Accuracy 82.97999572753906.
Iteration 3500. Test Loss 0.48858025670051575. Test Accuracy 83.3499984741211.
Starting epoch 5
Iteration 4000. Test Loss 0.4967201054096222. Test Accuracy 81.66999816894531.
Iteration 4500. Test Loss 0.48247453570365906. Test Accuracy 82.97000122070312.
Starting epoch 6
Iteration 5000. Test Loss 0.48985275626182556. Test Accuracy 82.9000015258789.
Iteration 5500. Test Loss 0.4788465201854706. Test Accuracy 82.94999694824219.
Starting epoch 7
Iteration 6000. Test Loss 0.489

In [37]:
import torch

# Make some fake data for MineFeedForwardNN.
# 5 examples in the batch, each example has 'input_dim' (1024) features.
sample_mine_input = torch.randn(5, input_dim)

# Move the input tensor to GPU if GPU is available and being used
if using_GPU:
  sample_mine_input = sample_mine_input.cuda()

# Run the sample_mine_input through mine_ffnn_clf to get a probability distribution
# over our classes (6 classes as defined in MineFeedForwardNN).
sample_mine_predictions = mine_ffnn_clf(sample_mine_input)

print("Predicted probability distribution over classes from MineFeedForwardNN: ")
print(sample_mine_predictions)

Predicted probability distribution over classes from MineFeedForwardNN: 
tensor([[0.1735, 0.1764, 0.1691, 0.1621, 0.1551, 0.1637],
        [0.1753, 0.1667, 0.1751, 0.1571, 0.1624, 0.1633],
        [0.1761, 0.1652, 0.1621, 0.1711, 0.1746, 0.1508],
        [0.1704, 0.1677, 0.1780, 0.1475, 0.1659, 0.1706],
        [0.1732, 0.1724, 0.1747, 0.1501, 0.1781, 0.1516]], device='cuda:0',
       grad_fn=<SoftmaxBackward0>)


## **Test 2**

Apply the neural network **`MineFeedForwardNN`** for MNIST dataset for classification (10 classes). Train your model with batch size 32 with 10 epochs.

Some notes:
- define your own dataloader, optimizer
- instantiate your network with proper input size and number of classes
- make sure your model output is compatible with your criteria (NLLLoss)

In [33]:
from torchvision.datasets import MNIST

train_dataset = MNIST(root='./torchvision-data',
                      train=True,
                      transform=torchvision.transforms.ToTensor(),
                      download=True)

test_dataset = MNIST(root='./torchvision-data', train=False,
                     transform=torchvision.transforms.ToTensor())

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.10MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 134kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.28MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.0MB/s]


In [48]:
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F # Import F for functional operations like log_softmax
import torch.optim as optim
from torch.autograd import Variable # Still using Variable as it's present in the notebook context, though modern PyTorch often uses direct tensors.

# Number of epochs (passes through the dataset) to train the model for.
num_epochs = 10

# A counter for the number of gradient updates we've performed.
num_iter = 0

# --- Start of modifications for batch_size ---
# Define batch size for this training cell
batch_size = 32

# Re-instantiate DataLoaders with the new batch_size
train_dataloader = DataLoader(
    dataset=train_dataset, batch_size=batch_size, shuffle=True)

test_dataloader = DataLoader(
    dataset=test_dataset, batch_size=batch_size)
# --- End of modifications for batch_size ---

# Iterate `num_epochs` times.
for epoch in range(num_epochs):
  print("Starting epoch {}".format(epoch + 1))
  # Iterate over the train_dataloader, unpacking the images and labels
  for (images, labels) in train_dataloader:
    # Reshape images from (batch_size, 1, 28, 28) to (batch_size, 784), since
    # that's what our model expects. Remember that -1 does shape inference!
    reshaped_images = images.view(-1, 784)

    # Wrap reshaped_images and labels in Variables, if not already tensors
    # Note: In modern PyTorch, direct tensors are often used without explicit Variable wrapping for this.
    reshaped_images = Variable(reshaped_images)
    labels = Variable(labels)

    # If we're using the GPU, move reshaped_images and labels to the GPU.
    if using_GPU:
      reshaped_images = reshaped_images.cuda()
      labels = labels.cuda()

    # Run the forward pass through the model to get predicted raw logits.
    # predicted shape: (batch_size, 10) (since there are 10 classes)
    predicted_logits = mine_ffnn_clf(reshaped_images)

    # Apply log_softmax to get log-probabilities for NLLLoss
    predicted = F.log_softmax(predicted_logits, dim=-1)

    # --- Added log function to log predicted values ---
    print(f"Epoch {epoch+1}, Iteration {num_iter + 1}: Predicted Log-Probabilities (first 5 samples):\n{predicted[:5]}")
    # --- End of added log function ---

    # Calculate the loss
    batch_loss = nll_criterion(predicted, labels)

    # Clear the gradients as we prepare to backprop.
    ffnn_optimizer.zero_grad()

    # Backprop (backward pass), which calculates gradients.
    batch_loss.backward()

    # Take a gradient step to update parameters.
    ffnn_optimizer.step()

    # Increment gradient update counter.
    num_iter += 1

    # Calculate test set loss and accuracy every 500 gradient updates
    # It's standard to have this as a separate evaluate function, but
    # we'll place it inline for didactic purposes.
    if num_iter % 500 == 0:
      # Set model to eval mode, which turns off dropout.
      fashionmnist_ffnn_clf.eval()
      # Counters for the num of examples we get right / total num of examples.
      num_correct = 0
      total_examples = 0
      total_test_loss = 0

      # Disable gradient calculations during evaluation
      with torch.no_grad():
        # Iterate over the test dataloader
        for (test_images, test_labels) in test_dataloader:
          # Reshape images from (batch_size, 1, 28, 28) to (batch_size, 784) again
          reshaped_test_images = test_images.view(-1, 784)

          # If we're using the GPU, move tensors to the GPU.
          if using_GPU:
            reshaped_test_images = reshaped_test_images.cuda()
            test_labels = test_labels.cuda()

          # Run the forward pass to get predicted distribution (logits).
          predicted_logits_eval = fashionmnist_ffnn_clf(reshaped_test_images)
          predicted_eval = F.log_softmax(predicted_logits_eval, dim=-1) # Apply log_softmax for NLLLoss

          # Calculate loss for this test batch. This is averaged, so multiply
          # by the number of examples in batch to get a total.
          total_test_loss += nll_criterion(
              predicted_eval, test_labels).item() * test_labels.size(0) # Using .item() for scalar tensors

          # Get predicted labels (argmax)
          _, predicted_labels = torch.max(predicted_eval, 1)

          # Count the number of examples in this batch
          total_examples += test_labels.size(0)

          # Count the total number of correctly predicted labels.
          # predicted == labels generates a ByteTensor in indices where
          # predicted and labels match, so we can sum to get the num correct.
          num_correct += torch.sum(predicted_labels == test_labels).item() # Using .item() for scalar tensors
      accuracy = 100 * num_correct / total_examples
      average_test_loss = total_test_loss / total_examples
      print("Iteration {}. Test Loss {}. Test Accuracy {} ".format(
          num_iter, average_test_loss, accuracy))
      # Set the model back to train mode, which activates dropout again.
      fashionmnist_ffnn_clf.train()

Starting epoch 1


RuntimeError: mat1 and mat2 shapes cannot be multiplied (32x784 and 1024x1024)

In [46]:
if using_GPU:
    mine_ffnn_clf = mine_ffnn_clf.cuda()

# Check if the Module is on GPU

print("Model on GPU?:")
print(next(mine_ffnn_clf.parameters()).is_cuda)

Model on GPU?:
True
